In [ ]:
#Kiểm tra môi trường
import torch
print(torch.__version__, torch.version.cuda)
print("GPU:", torch.cuda.is_available())

2.11.0+cu128 12.8
GPU: True


In [ ]:
#Cài thư viện đồ thị đúng phiên bản torch của Colab
v = torch.__version__.split('+')[0]
c = torch.version.cuda.replace('.', '')

!pip install torch-scatter torch-sparse --only-binary=:all: -f https://data.pyg.org/whl/torch-{v}+cu{c}.html
!pip install torch_geometric scipy scikit-learn pandas

import torch_scatter, torch_sparse
print("torch_scatter:", torch_scatter.__version__)
print("torch_sparse:", torch_sparse.__version__)

Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 36.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 50.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 76.8 MB/s eta 0:00:00
torch_scatter: 2.1.2+pt211cu128
torch_sparse: 0.6.18+pt211cu128


In [ ]:
!git clone https://github.com/seongjunyun/Graph_Transformer_Networks.git
!cp Graph_Transformer_Networks/*.py /content/

import os
need = ['main.py','model_gtn.py','model_fastgtn.py','inits.py','utils.py']
missing = [f for f in need if not os.path.exists(f)]
print("Thiếu:", missing if missing else "Không thiếu gì, OK!")

Cloning into 'Graph_Transformer_Networks'...
remote: Enumerating objects: 137, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 137 (delta 36), reused 33 (delta 33), pack-reused 93 (from 1)
Receiving objects: 100% (137/137), 18.83 MiB | 18.29 MiB/s, done.
Resolving deltas: 100% (69/69), done.
Thiếu: Không thiếu gì, OK!


In [ ]:
import os, pickle
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
import shutil # For removing directories
import pandas as pd

# Ensure data_raw directory is clean or recreated
if os.path.exists('data_raw'):
    shutil.rmtree('data_raw')
os.makedirs('data_raw', exist_ok=True)

# Clean up previous clone if it exists to avoid 'destination path already exists' error
repo_name = "Datasets-for-Heterogeneous-Graph"
if os.path.exists(repo_name):
    shutil.rmtree(repo_name)

# Clone the repository containing the dataset
repo_url = "https://github.com/Jhy1993/Datasets-for-Heterogeneous-Graph.git"
!git clone {repo_url}

# List contents of the cloned repository to debug path issue
print(f"Listing contents of {repo_name}:")
!ls -R {repo_name}

# Define paths for raw Yelp_Business files
yelp_business_dir = os.path.join(repo_name, 'Yelp_Business')
user_business_path = os.path.join(yelp_business_dir, 'user_business.dat')
bus_cate_path = os.path.join(yelp_business_dir, 'bus_cate.txt')
user_user_path = os.path.join(yelp_business_dir, 'user_user.dat')
business_city_path = os.path.join(yelp_business_dir, 'business_city.dat') # For potential business features

print("Đọc dữ liệu thô từ Yelp_Business...")
# Read raw data with appropriate separators and names
# user_business.dat: user_id, business_id, rating (tab-separated)
user_business = pd.read_csv(user_business_path, sep='\t', header=None, names=['user_id', 'business_id', 'rating'])

# bus_cate.txt: business_id, category_id (space-separated, 2 columns)
business_category = pd.read_csv(bus_cate_path, sep=' ', header=None, names=['business_id', 'category_id'])

# user_user.dat: user_id_1, user_id_2, trust, unknown_col (tab-separated)
user_user = pd.read_csv(user_user_path, sep='\t', header=None, names=['user_id_1', 'user_id_2', 'trust', 'unknown_col'])

# business_city.dat: business_id, city_id, count (tab-separated)
business_city = pd.read_csv(business_city_path, sep='\t', header=None, names=['business_id', 'city_id', 'count'])

# Map IDs to consecutive integers
# Collecting all unique user IDs from all relevant dataframes
all_user_ids = pd.concat([
    user_business['user_id'],
    user_user['user_id_1'],
    user_user['user_id_2']
]).unique()

# Collecting all unique business IDs
all_business_ids = pd.concat([
    user_business['business_id'],
    business_category['business_id'],
    business_city['business_id']
]).unique()

# Collecting all unique category IDs
all_category_ids = business_category['category_id'].unique()

# Collecting all unique city IDs
all_city_ids = business_city['city_id'].unique()
# Filter out NaN values from all_city_ids if any, before creating city_to_idx
all_city_ids = all_city_ids[pd.notna(all_city_ids)]

user_to_idx = {uid: i for i, uid in enumerate(all_user_ids)}
business_to_idx = {bid: i for i, bid in enumerate(all_business_ids)}
category_to_idx = {cid: i for i, cid in enumerate(all_category_ids)}
city_to_idx = {cid: i for i, cid in enumerate(all_city_ids)}

# Adjust dataframes with mapped indices
user_business['user_idx'] = user_business['user_id'].map(user_to_idx)
user_business['business_idx'] = user_business['business_id'].map(business_to_idx)

business_category['business_idx'] = business_category['business_id'].map(business_to_idx)
business_category['category_idx'] = business_category['category_id'].map(category_to_idx)

user_user['user_1_idx'] = user_user['user_id_1'].map(user_to_idx)
user_user['user_2_idx'] = user_user['user_id_2'].map(user_to_idx)

business_city['business_idx'] = business_city['business_id'].map(business_to_idx)
business_city['city_idx'] = business_city['city_id'].map(city_to_idx)

# Drop rows with NaN indices due to mapping if any ID was not found
# (should be rare if all unique IDs are collected properly, but good for robustness)
user_business.dropna(subset=['user_idx', 'business_idx'], inplace=True)
business_category.dropna(subset=['business_idx', 'category_idx'], inplace=True)
user_user.dropna(subset=['user_1_idx', 'user_2_idx'], inplace=True)
business_city.dropna(subset=['business_idx', 'city_idx'], inplace=True)

# Convert indices to integer type after dropping NaNs
user_business['user_idx'] = user_business['user_idx'].astype(int)
user_business['business_idx'] = user_business['business_idx'].astype(int)
business_category['business_idx'] = business_category['business_idx'].astype(int)
business_category['category_idx'] = business_category['category_idx'].astype(int)
user_user['user_1_idx'] = user_user['user_1_idx'].astype(int)
user_user['user_2_idx'] = user_user['user_2_idx'].astype(int)
business_city['business_idx'] = business_city['business_idx'].astype(int)
business_city['city_idx'] = business_city['city_idx'].astype(int)

n_u = len(all_user_ids)
n_b = len(all_business_ids)
n_c = len(all_category_ids)
n_city = len(all_city_ids)

# Recalculate N, off_b, off_c based on actual counts
N = n_u + n_b + n_c # Total nodes for users, businesses, categories
off_b, off_c = n_u, n_u + n_b

print(f"User nodes: {n_u} | Business nodes: {n_b} | Category nodes: {n_c} | Total nodes: {N}")

# ---- Features ----
# User features: Use identity matrix as no explicit features are provided
user_features = np.eye(n_u, dtype=np.float32)

# Business features: One-hot encode categories and cities
# Create a sparse matrix for business-category features
# Handle case where business_category might be empty after dropna
if not business_category.empty:
    business_cate_feat = csr_matrix((np.ones(len(business_category)),
                                   (business_category['business_idx'], business_category['category_idx'])),
                                  shape=(n_b, n_c)).toarray().astype(np.float32)
else:
    business_cate_feat = np.zeros((n_b, n_c), dtype=np.float32)

# Create a sparse matrix for business-city features
# Handle case where business_city might be empty after dropna
if not business_city.empty:
    business_city_feat = csr_matrix((np.ones(len(business_city)),
                                   (business_city['business_idx'], business_city['city_idx'])),
                                  shape=(n_b, n_city)).toarray().astype(np.float32)
else:
    business_city_feat = np.zeros((n_b, n_city), dtype=np.float32)

business_features = np.hstack([business_cate_feat, business_city_feat])

# Category features: Use identity matrix
category_features = np.eye(n_c, dtype=np.float32)

# Determine max feature dimension for padding
max_dim = max(user_features.shape[1], business_features.shape[1], category_features.shape[1])

def pad(x, target_dim):
    out = np.zeros((x.shape[0], target_dim), dtype=np.float32)
    out[:, :x.shape[1]] = x
    return out

features = np.vstack([
    pad(user_features, max_dim),      # users
    pad(business_features, max_dim),  # businesses
    pad(category_features, max_dim)   # categories
])

# ---- Edges (user-business, business-category, user-user and their transposes) ----
def adj(rows, cols, num_nodes):
    # Ensure rows and cols are numpy arrays for csr_matrix
    rows_np = np.array(rows)
    cols_np = np.array(cols)
    # Filter out invalid indices if any (shouldn't happen with proper ID mapping)
    valid_indices = (rows_np >= 0) & (rows_np < num_nodes) & (cols_np >= 0) & (cols_np < num_nodes)
    rows_np = rows_np[valid_indices]
    cols_np = cols_np[valid_indices]

    if len(rows_np) == 0:
        return csr_matrix((num_nodes, num_nodes), dtype=np.float32)
    return csr_matrix((np.ones(len(rows_np)), (rows_np, cols_np)), shape=(num_nodes, num_nodes))

# User-Business interactions (directed u -> b, b -> u)
# Ensure indices are valid and dataframes are not empty
u_idx_ub, b_idx_ub = [], []
if not user_business.empty:
    u_idx_ub = user_business['user_idx'].values
    b_idx_ub = user_business['business_idx'].values + off_b

# Business-Category (directed b -> c, c -> b)
b_idx_bc, c_idx_bc = [], []
if not business_category.empty:
    b_idx_bc = business_category['business_idx'].values + off_b
    c_idx_bc = business_category['category_idx'].values + off_c

# User-User (directed u -> u)
u_idx_uu1, u_idx_uu2 = [], []
if not user_user.empty:
    u_idx_uu1 = user_user['user_1_idx'].values
    u_idx_uu2 = user_user['user_2_idx'].values

edges = [
    adj(u_idx_ub, b_idx_ub, N), adj(b_idx_ub, u_idx_ub, N), # User-Business and Business-User
    adj(b_idx_bc, c_idx_bc, N), adj(c_idx_bc, b_idx_bc, N), # Business-Category and Category-Business
    adj(u_idx_uu1, u_idx_uu2, N), adj(u_idx_uu2, u_idx_uu1, N) # User-User (assuming symmetric trust)
]

# ---- Labels (PLACEHOLDER: REQUIRES USER INPUT) ----
# The original YelpChi.mat had 'label' for fraud detection.
# Since there's no explicit label file in this 'Yelp_Business' repository,
# we need to ask the user how to define the labels.
# For now, creating a dummy label array for demonstration or until user provides input.
# If we assume a binary classification task for users, for example.

# Create dummy labels for users for now (e.g., two classes, balanced)
# In a real scenario, this would come from a specific column/file.
labels_all = np.random.randint(0, 2, n_u) # Dummy binary labels for users
print(f"Dummy labels created for {n_u} users. Label distribution: {np.bincount(labels_all)}")

all_idx = np.arange(n_u) # Assuming labels are for user nodes
train_size = 0.4
valid_size = 0.33

if n_u > 0: # Ensure there are users to split
    train_idx, rest_idx, train_y, rest_y = train_test_split(all_idx, labels_all, train_size=train_size, stratify=labels_all, random_state=42)
    valid_idx, test_idx, valid_y, test_y = train_test_split(rest_idx, rest_y, train_size=valid_size, stratify=rest_y, random_state=42)
else:
    train_idx, train_y = np.array([]), np.array([])
    valid_idx, valid_y = np.array([]), np.array([])
    test_idx, test_y = np.array([]), np.array([])

labels = [np.vstack((train_idx, train_y)).T.astype(int),
          np.vstack((valid_idx, valid_y)).T.astype(int),
          np.vstack((test_idx, test_y)).T.astype(int)]

os.makedirs('data/Yelp_Business', exist_ok=True)
with open('data/Yelp_Business/node_features.pkl', 'wb') as f: pickle.dump(features, f)
with open('data/Yelp_Business/edges.pkl', 'wb') as f: pickle.dump(edges, f)
with open('data/Yelp_Business/labels.pkl', 'wb') as f: pickle.dump(labels, f)
print("Đã lưu data/Yelp_Business/")

# Clean up the cloned repository
if os.path.exists(repo_name):
    shutil.rmtree(repo_name)


Cloning into 'Datasets-for-Heterogeneous-Graph'...
remote: Enumerating objects: 113, done.
remote: Total 113 (delta 0), reused 0 (delta 0), pack-reused 113 (from 1)
Receiving objects: 100% (113/113), 59.50 MiB | 17.67 MiB/s, done.
Resolving deltas: 100% (14/14), done.
Listing contents of Datasets-for-Heterogeneous-Graph:
Datasets-for-Heterogeneous-Graph:
 ACM_1	  Aminer	'Douban Movie'	 Movielens   Yelp_Business
 ACM_3	  DBLP_1	 LastFM		 README.md
 Amazon  'Douban Book'	 ml-1m		 Yelp_2

Datasets-for-Heterogeneous-Graph/ACM_1:
ACMdataset.mat

Datasets-for-Heterogeneous-Graph/ACM_3:
ACM.mat

Datasets-for-Heterogeneous-Graph/Amazon:
item_brand.dat	item_category.dat  item_view.dat  README  user_item.dat

Datasets-for-Heterogeneous-Graph/Aminer:
author2id.txt  label2id.txt	 paper_conference.txt  paper_type.txt  README
conf2id.txt    paper_author.txt  paper_label.txt       paper_year.txt

Datasets-for-Heterogeneous-Graph/DBLP_1:
author_label.dat   paper_author.dat	 paper_type.dat
author_map_i

In [ ]:
import pickle
with open('data/Yelp_Business/node_features.pkl','rb') as f: X = pickle.load(f)
with open('data/Yelp_Business/edges.pkl','rb') as f: E = pickle.load(f)
with open('data/Yelp_Business/labels.pkl','rb') as f: L = pickle.load(f)

print("features:", X.shape)                    # (Tổng số nodes, max_dim)
print("số loại cạnh:", len(E), "| shape:", E[0].shape)  # 6 | (Tổng số nodes, Tổng số nodes)
print("số cạnh mỗi loại:", [int(e.nnz) for e in E])
print("train/val/test:", [len(l) for l in L])  # Số lượng nhãn train/val/test

features: (31035, 16239)
số loại cạnh: 6 | shape: (31035, 31035)
số cạnh mỗi loại: [198397, 198397, 39955, 39955, 158590, 158590]
train/val/test: [6495, 3215, 6529]


In [ ]:
with open('main.py', 'r') as f:
    content = f.read()

# Vá 1: f1_score đã bị xóa khỏi torch_geometric bản mới
content = content.replace(
    "from torch_geometric.utils import f1_score, add_self_loops",
    "from torch_geometric.utils import add_self_loops"
)
content = content.replace(
    "train_f1 = torch.mean(f1_score(torch.argmax(y_train.detach(),dim=1), train_target, num_classes=num_classes)).cpu().numpy()",
    "train_f1 = sk_f1_score(train_target.detach().cpu(), torch.argmax(y_train.detach(),dim=1).cpu(), average='macro')"
)
content = content.replace(
    "val_f1 = torch.mean(f1_score(torch.argmax(y_valid,dim=1), valid_target, num_classes=num_classes)).cpu().numpy()",
    "val_f1 = sk_f1_score(valid_target.detach().cpu(), torch.argmax(y_valid,dim=1).cpu(), average='macro')"
)
content = content.replace(
    "test_f1 = torch.mean(f1_score(torch.argmax(y_test,dim=1), test_target, num_classes=num_classes)).cpu().numpy()",
    "test_f1 = sk_f1_score(test_target.detach().cpu(), torch.argmax(y_test,dim=1).cpu(), average='macro')"
)

# Vá 2: đường dẫn ../data -> data
content = content.replace("'../data/%s/", "'data/%s/")

with open('main.py', 'w') as f:
    f.write(content)

# Kiểm tra luôn
with open('main.py') as f:
    ok = 'from torch_geometric.utils import f1_score' not in f.read()
print("Đã vá thành công:", ok)

Đã vá thành công: True


In [ ]:
!python main.py --dataset Yelp_Business --model FastGTN --num_layers 3 --epoch 200 --lr 0.05 --channel_agg mean --num_channels 2

Namespace(model='FastGTN', dataset='Yelp_Business', epoch=200, node_dim=64, num_channels=2, lr=0.05, weight_decay=0.001, num_layers=3, runs=10, channel_agg='mean', remove_self_loops=False, non_local=False, non_local_weight=0, beta=0, K=1, pre_train=False, num_FastGTN_layers=1)
/content/model_fastgtn.py:158: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  mat_a = torch.sparse_coo_tensor(a_edge, a_value, (num_nodes, num_nodes)).to(a_edge.device)
Run 0
--------------------Best Result-------------------------
Train - Loss: 0.6930, Macro_F1: 0.5612, Micro_F1: 0.5843
Valid - Loss: 0.6934, Macro_F1: 0.5049, Micro_F1: 0.5095
Test - Loss: 0.6930, Mac

In [ ]:
import pickle, torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import f1_score

# Load đúng dữ liệu và split đã dùng cho FastGTN
with open('data/Yelp_Business/node_features.pkl','rb') as f: X = pickle.load(f)
with open('data/Yelp_Business/labels.pkl','rb') as f: L = pickle.load(f)

# MLP chỉ thấy feature user (giới tính + nghề nghiệp) — KHÔNG thấy đồ thị
X = torch.tensor(X, dtype=torch.float32).cuda()
tr_i, tr_y = torch.tensor(L[0][:,0]).cuda(), torch.tensor(L[0][:,1]).cuda()
va_i, va_y = torch.tensor(L[1][:,0]).cuda(), torch.tensor(L[1][:,1]).cuda()
te_i, te_y = torch.tensor(L[2][:,0]).cuda(), torch.tensor(L[2][:,1]).cuda()

# Determine input and output dimensions dynamically
input_dim = X.shape[1] # Number of features
num_classes = len(torch.unique(tr_y)) # Number of unique classes in training labels

def run_once(seed):
    torch.manual_seed(seed)
    mlp = nn.Sequential(nn.Linear(input_dim,64), nn.ReLU(), nn.Dropout(0.5),
                        nn.Linear(64,64), nn.ReLU(), nn.Dropout(0.5),
                        nn.Linear(64,num_classes)).cuda()
    opt = torch.optim.Adam(mlp.parameters(), lr=0.005, weight_decay=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    best_val, best_test_ma, best_test_mi = 0, 0, 0
    for ep in range(200):
        mlp.train(); opt.zero_grad()
        loss = loss_fn(mlp(X[tr_i]), tr_y)
        loss.backward(); opt.step()
        mlp.eval()
        with torch.no_grad():
            va_pred = mlp(X[va_i]).argmax(1).cpu()
            te_pred = mlp(X[te_i]).argmax(1).cpu()
        val_f1 = f1_score(va_y.cpu(), va_pred, average='macro')
        if val_f1 > best_val:
            best_val = val_f1
            best_test_ma = f1_score(te_y.cpu(), te_pred, average='macro')
            best_test_mi = f1_score(te_y.cpu(), te_pred, average='micro')
    return best_test_ma, best_test_mi

results = [run_once(s) for s in range(10)]
ma = np.array([r[0] for r in results]); mi = np.array([r[1] for r in results])
print(f"MLP baseline (không đồ thị) — Test Macro_F1: {ma.mean():.4f}+{ma.std():.4f}, Micro_F1: {mi.mean():.4f}+{mi.std():.4f}")
print(f"FastGTN (có đồ thị)        — Test Macro_F1: 0.4734+0.0417, Micro_F1:0.5063+0.0050)")

MLP baseline (không đồ thị) — Test Macro_F1: 0.4513+0.0635, Micro_F1: 0.5018+0.0042
FastGTN (có đồ thị)        — Test Macro_F1: 0.4734+0.0417, Micro_F1:0.5063+0.0050)
